In [271]:
import pandas as pd
from sklearn.pipeline import Pipeline
import numpy as np
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\raeve\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\raeve\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [272]:
data_one = pd.read_csv('twitter_training.csv')
data_two = pd.read_csv('data.csv')
data_three = pd.read_csv('stock_data.csv')
data_four = pd.read_csv('sentiment_analysis.csv')

In [273]:
data_one = data_one.drop(columns=['2401', "Borderlands"])
data_one = data_one.rename(columns={'Positive': 'Sentiment', 'im getting on borderlands and i will murder you all ,': 'Text'})
data_one = data_one.dropna()

In [274]:
data_one.head()
data_one['Sentiment'] = data_one['Sentiment'].str.lower()
data_one['Sentiment'].unique()

array(['positive', 'neutral', 'negative', 'irrelevant'], dtype=object)

In [275]:
data_two = data_two.rename(columns={'Sentence': 'Text'})
data_two = data_two[["Sentiment", "Text"]]
data_two = data_two.dropna()

In [276]:
data_two['Sentiment'].unique()

array(['positive', 'negative', 'neutral'], dtype=object)

In [277]:
data_three = data_three[["Sentiment", "Text"]]
data_three["Sentiment"] = data_three["Sentiment"].replace({1: "positive", -1: "negative"})
data_three['Sentiment'].unique()

array(['positive', 'negative'], dtype=object)

In [278]:
data_four.head()
data_four = data_four.drop(columns=['Year', 'Month', 'Day', 'Time of Tweet', 'Platform'])
data_four = data_four.rename(columns={'text': 'Text', 'sentiment': 'Sentiment'})
data_four["Sentiment"].unique()
data_four.isnull().sum()

Text         0
Sentiment    0
dtype: int64

In [279]:
df = pd.concat([data_one, data_two, data_three, data_four], axis=0)

In [280]:
df.isna().sum()

Sentiment    0
Text         0
dtype: int64

In [281]:
df = df[df['Sentiment'] != 'irrelevant']

In [282]:
df['Sentiment'].unique()

array(['positive', 'neutral', 'negative'], dtype=object)

PreProcess

In [283]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [284]:
def pre_process(text):
    # Handle potential non-string/null values
    if not isinstance(text, str):
        return ""

    # 2. Cleaning & Normalization
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Better URL removal
    text = re.sub(r'@\w+', '', text)                                       # Remove mentions
    text = re.sub(r'\d+', '', text)                                        # Remove numbers
    text = re.sub(r'[^\w\s]', '', text)                                    # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()                               # Remove extra whitespace

    # 3. Tokenization
    tokens = word_tokenize(text)

    # 4. Filtering & Lemmatization in one efficient list comprehension
    # We check if the word is a stopword BEFORE lemmatizing to save CPU cycles.
    cleaned_tokens = [
        lemmatizer.lemmatize(word) 
        for word in tokens 
        if word not in stop_words and len(word) > 2 # Also removes tiny "noise" words
    ]

    return ' '.join(cleaned_tokens)

# 5. Apply with a Progress Bar (Optional but helpful for large datasets)
# If using a massive dataframe, consider: df['Text'].map(pre_process) 
# .map() is generally faster than .apply() for Series.
df['Clean_Text'] = df['Text'].astype(str).apply(pre_process)

In [285]:
le = LabelEncoder()

df['Sentiment'] = le.fit_transform(df['Sentiment'])

In [286]:
# vectorize
vectorizer = TfidfVectorizer(max_features=20000,
    ngram_range=(1,2),
    stop_words='english')
X = vectorizer.fit_transform(df['Clean_Text'])
y = df['Sentiment']

In [287]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [288]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [289]:
svc = LinearSVC(
    C=1.0,
    random_state=42
)

svc.fit(X_train, y_train)

# Predictions
y_pred = svc.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8367346938775511

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.83      0.85      5208
           1       0.84      0.80      0.82      4251
           2       0.81      0.87      0.84      5192

    accuracy                           0.84     14651
   macro avg       0.84      0.83      0.84     14651
weighted avg       0.84      0.84      0.84     14651


Confusion Matrix:
[[4336  321  551]
 [ 311 3394  546]
 [ 334  329 4529]]


In [290]:
print("Train Accuracy:", svc.score(X_train, y_train))
print("Test Accuracy:", svc.score(X_test, y_test))

Train Accuracy: 0.9109742154570741
Test Accuracy: 0.8367346938775511


In [299]:
param_grid = {
    # LinearSVC tuning
    'C': [2, 3, 4, 5],
    'loss': ['hinge', 'squared_hinge'],
    'dual': [True, False],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    estimator=LinearSVC(),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

Fitting 3 folds for each of 32 candidates, totalling 96 fits


c:\Users\raeve\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
24 fits failed out of a total of 96.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
24 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\raeve\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\raeve\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\raeve\AppData\Local\Programs\Python\Python39\lib\site-packages\sk

GridSearchCV(cv=3, estimator=LinearSVC(), n_jobs=-1,
             param_grid={'C': [2, 3, 4, 5], 'class_weight': [None, 'balanced'],
                         'dual': [True, False],
                         'loss': ['hinge', 'squared_hinge']},
             scoring='accuracy', verbose=2)

In [300]:
print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Params: {'C': 2, 'class_weight': None, 'dual': False, 'loss': 'squared_hinge'}
Best CV Score: 0.8187403351972179


In [301]:
best_model = grid.best_estimator_

In [302]:
y_pred = best_model.predict(X_test)

In [303]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.83      0.85      5208
           1       0.84      0.80      0.82      4251
           2       0.81      0.87      0.84      5192

    accuracy                           0.84     14651
   macro avg       0.84      0.84      0.84     14651
weighted avg       0.84      0.84      0.84     14651



In [304]:
import joblib

# save trained model
joblib.dump(best_model, "sentiment_model.pkl")

# save TF-IDF separately (if you used manual vectorizer)
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']